In [ ]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import math


LOOKBACK = 168
FORECAST = 24
BATCH_SIZE = 128
EPOCHS = 100
LR = 0.001

TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.10
PATIENCE = 15
TARGET_BUILDINGS = [
    'Hog_office_Shon',
    'Hog_office_Betsy',
    'Hog_parking_Jean',
    'Hog_office_Mary',
    'Hog_office_Miriam',
    'Hog_public_Kevin',
    'Hog_office_Valda',
    'Hog_office_Shawnna',
    'Robin_lodging_Janie',
    'Hog_education_Casandra',
    'Hog_services_Adrianna',
    'Hog_office_Denita',
    'Hog_public_Crystal',
    'Bear_education_Iris',
    'Hog_office_Cordelia',
    'Bear_education_Elise',
    'Bear_education_Thuy',
    'Bear_education_Darrell',
    'Wolf_retail_Toshia',
    'Hog_office_Mari',
    'Wolf_education_Cheryl',
    'Bear_education_Bob',
    'Bear_education_Marie',
    'Bear_education_Shelley',
    'Bear_public_Rayna',
    'Bear_public_Valorie',
    'Wolf_education_Laurinda',
    'Wolf_public_Norma',
    'Bear_education_Paola',
    'Fox_parking_Lynelle',
    'Rat_office_Jamal',
    'Peacock_education_Robbie',
    'Wolf_education_Katie',
    'Fox_education_Ray',
    'Fox_office_Karima',
    'Wolf_education_Eulalia',
    'Bear_education_Danna',
    'Wolf_office_Nadia',
    'Wolf_education_Loren',
    'Fox_office_Clayton',
    'Bear_education_Leonel',
    'Fox_office_Thelma',
    'Wolf_office_Haydee',
    'Wolf_education_Tori',
    'Hog_education_Rachael',
    'Hog_office_Alexis',
    'Hog_lodging_Francisco',
    'Wolf_education_Vivian',
    'Fox_education_Long',
    'Fox_office_Carson',
    'Wolf_office_Emanuel',
    'Fox_public_Bart',
    'Fox_office_Susanne',
    'Fox_utility_Marian',
    'Hog_parking_Shannon',
    'Bear_education_Katy',
    'Wolf_office_Bobbie',
    'Hog_office_Migdalia',
    'Hog_industrial_Mariah',
    'Hog_office_Patrick',
    'Fox_food_Francesco',
    'Hog_industrial_Jeremy',
    'Hog_industrial_Joanne',
    'Fox_education_Kendrick',
    'Wolf_education_Josefa',
    'Peacock_office_Annie',
    'Hog_office_Richelle',
    'Fox_education_Sterling',
    'Hog_education_Robert',
    'Fox_education_Kim',
    'Fox_public_Rhonda',
    'Peacock_education_Patience',
    'Peacock_office_Julian',
    'Peacock_office_Effie',
    'Fox_education_Shawanda',
    'Fox_education_Elois',
    'Hog_office_Sherrie',
    'Fox_education_Heriberto',
    'Fox_education_Tamika',
    'Hog_office_Shawna',
    'Fox_education_Kris',
    'Hog_office_Almeda',
    'Wolf_office_Cary',
    'Hog_education_Janell',
    'Fox_education_Miguelina',
    'Hog_public_Gerard',
    'Wolf_education_Miguel',
    'Fox_office_Yong',
    'Hog_lodging_Mauricio',
    'Fox_office_Brandy',
    'Bobcat_education_Alissa',
    'Lamb_education_Lucas',
    'Lamb_education_Bettye',
    'Lamb_education_Princess',
    'Lamb_education_Antonio',
    'Lamb_education_Kayla',
    'Lamb_education_Harold',
    'Lamb_education_Dolly',
    'Lamb_education_Gabriel',
    'Lamb_education_Rick',
    'Lamb_education_Gabrielle',
    'Lamb_education_Mae',
    'Lamb_education_Moses',
    'Lamb_education_Freddy',
    'Lamb_education_Sara',
    'Lamb_education_Norris',
    'Lamb_education_Isabelle',
    'Lamb_education_Darrel',
    'Lamb_education_Sabrina',
    'Lamb_education_Phil',
    'Lamb_education_Ruben',
    'Lamb_education_Mckenzie',
    'Lamb_education_Patsy',
    'Lamb_education_Faith',
    'Lamb_office_Vasiliki',
    'Lamb_office_Velma',
    'Lamb_office_Raymond',
    'Lamb_office_Caitlin',
    'Lamb_office_Bertha',
    'Lamb_office_Jo',
    'Lamb_office_Stefani',
    'Lamb_office_Callie',
    'Lamb_office_Gerardo',
    'Lamb_office_Peggy',
    'Robin_office_Wai',
    'Robin_office_Zelma',
    'Bobcat_office_Nikita',
    'Eagle_office_Henriette',
    'Eagle_office_Francis',
    'Eagle_office_Nereida',
    'Eagle_office_Remedios',
    'Eagle_office_Phyllis',
    'Robin_office_Donald',
    'Lamb_office_William',
    'Lamb_assembly_Isa',
    'Lamb_food_Sylvia',
    'Lamb_health_Ken',
    'Lamb_lodging_Burt',
    'Lamb_industrial_Venessa',
    'Lamb_other_Katharina',
    'Hog_parking_Bernardo',
    'Lamb_public_Grace',
    'Cockatoo_religion_Diedre',
    'Fox_retail_Manie',
    'Hog_services_Kerrie',
    'Cockatoo_science_Rex',
    'Cockatoo_utility_Sherri',
    'Lamb_warehouse_Allan',
    'Lamb_assembly_Dorathy',
    'Eagle_food_Kay'
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f'Seed setat: {SEED}')

In [ ]:
import pandas as pd
import torch.nn as nn


class BaselineLSTM(nn.Module):
    def __init__(self, num_math_features, hidden_dim=64):
        super(BaselineLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=num_math_features,
                            hidden_size=hidden_dim,
                            num_layers=1,
                            batch_first=True,
                            bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, FORECAST)
        )

    def forward(self, math_input):
        _, (hidden_state, _) = self.lstm(math_input)
        math_vector = torch.cat((hidden_state[-2], hidden_state[-1]), dim=1)
        return self.fc(math_vector)


In [ ]:
def build_base_series_with_weather(df_raw, building_col, df_metadata, df_weather):
    df_b = df_raw[["timestamp", building_col]].copy()
    df_b.rename(columns={building_col: "consumed_energy"}, inplace=True)

    if "timestamp" in df_b.columns:
        df_b.set_index("timestamp", inplace=True)

    df_b = df_b[~df_b.index.duplicated(keep="first")]
    df_b.sort_index(inplace=True)

    site_id = None
    if df_metadata is not None and not df_metadata.empty:
        meta_row = df_metadata[df_metadata["building_id"] == building_col]
        if not meta_row.empty:
            site_id = meta_row.iloc[0].get("site_id", None)

    if site_id is not None and df_weather is not None and not df_weather.empty:
        df_w = df_weather[df_weather["site_id"] == site_id].copy()
        if "timestamp" in df_w.columns:
            df_w.set_index("timestamp", inplace=True)
        df_w = df_w[["airTemperature", "dewTemperature", "cloudCoverage"]]
        df_b = df_b.join(df_w, how="left")
    else:
        df_b["airTemperature"] = np.nan
        df_b["dewTemperature"] = np.nan
        df_b["cloudCoverage"] = np.nan

    return df_b


def apply_split_cleaning_and_features(df_b, median_consumption):
    df_b = df_b.copy()

    for col in ["airTemperature", "dewTemperature", "cloudCoverage"]:
        df_b[col] = df_b[col].ffill(limit=24)

    MIN_FLAT_RUN = 4
    diff_abs = df_b["consumed_energy"].diff().abs()
    is_flat = (diff_abs < 0.001) & df_b["consumed_energy"].notna()
    flat_run_id = (is_flat != is_flat.shift()).cumsum()
    flat_run_len = is_flat.groupby(flat_run_id).transform("sum")
    df_b.loc[is_flat & (flat_run_len >= MIN_FLAT_RUN), "consumed_energy"] = np.nan

    prag_minim = median_consumption * 0.10
    df_b.loc[df_b["consumed_energy"] <= prag_minim, "consumed_energy"] = np.nan

    df_b["consumed_energy"] = df_b["consumed_energy"].interpolate(
        method="linear", limit=6, limit_direction="forward"
    )
    df_b["consumed_energy"] = df_b["consumed_energy"].ffill(limit=24)

    df_b["hour"] = df_b.index.hour
    df_b["day_of_week"] = df_b.index.dayofweek
    df_b["month"] = df_b.index.month

    df_b["is_weekend"] = df_b["day_of_week"].apply(lambda x: 1 if x >= 5 else 0)
    df_b["is_business_hours"] = df_b["hour"].apply(lambda x: 1 if 8 <= x <= 18 else 0)

    df_b["hour_sin"] = np.sin(2 * np.pi * df_b["hour"] / 24.0)
    df_b["hour_cos"] = np.cos(2 * np.pi * df_b["hour"] / 24.0)
    df_b["dow_sin"] = np.sin(2 * np.pi * df_b["day_of_week"] / 7.0)
    df_b["dow_cos"] = np.cos(2 * np.pi * df_b["day_of_week"] / 7.0)
    df_b["month_sin"] = np.sin(2 * np.pi * df_b["month"] / 12.0)
    df_b["month_cos"] = np.cos(2 * np.pi * df_b["month"] / 12.0)

    df_b["ma_6h"] = df_b["consumed_energy"].rolling(window=6, min_periods=1).mean()
    df_b["ma_24h"] = df_b["consumed_energy"].rolling(window=24, min_periods=1).mean()
    df_b["gradient"] = df_b["consumed_energy"].diff().fillna(0)

    df_b.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_b.dropna(inplace=True)
    return df_b


def create_windows(df_b):
    if len(df_b) < (LOOKBACK + FORECAST):
        return (
            np.empty((0, LOOKBACK, len(all_features))),
            np.empty((0, FORECAST)),
            np.empty((0,), dtype="datetime64[ns]")
        )
    X_math, y, ts = [], [], []
    idx = df_b.index
    feat_values = df_b[all_features].values
    energy_values = df_b["consumed_energy"].values
    window_span = pd.Timedelta(hours=LOOKBACK + FORECAST - 1)
    skipped = 0
    for i in range(len(df_b) - LOOKBACK - FORECAST + 1):
        if idx[i + LOOKBACK + FORECAST - 1] - idx[i] != window_span:
            skipped += 1
            continue
        X_math.append(feat_values[i : i + LOOKBACK])
        y.append(energy_values[i + LOOKBACK : i + LOOKBACK + FORECAST])
        ts.append(idx[i + LOOKBACK])
    if skipped:
        print(f"  Ferestre necontigue eliminate: {skipped} (gol temporal dupa curatare)")
    if len(X_math) == 0:
        return (
            np.empty((0, LOOKBACK, len(all_features))),
            np.empty((0, FORECAST)),
            np.empty((0,), dtype="datetime64[ns]")
        )
    return np.array(X_math), np.array(y), np.array(ts)


def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    wape = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) if np.sum(y_true) != 0 else 0
    r2 = r2_score(y_true, y_pred)
    max_err = np.max(np.abs(y_true - y_pred))
    return mae, rmse, wape, r2, max_err


In [ ]:
df_raw = pd.read_csv('data_set_genome\electricity.csv')
df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])

df_metadata = pd.read_csv('data_set_genome\metadata.csv')

df_weather = pd.read_csv('data_set_genome\weather.csv')
df_weather['timestamp'] = pd.to_datetime(df_weather['timestamp'])

if not df_weather.empty:
    df_weather = df_weather.sort_values(['site_id', 'timestamp'])


In [ ]:
features_to_scale = ['consumed_energy', 'ma_6h', 'ma_24h', 'gradient', 'airTemperature', 'dewTemperature', 'cloudCoverage']
all_features = ['consumed_energy', 'is_weekend', 'is_business_hours', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'ma_6h', 'ma_24h', 'gradient', 'airTemperature', 'dewTemperature', 'cloudCoverage']

precomputed_math = {}

def prepare_building_math_data(building):
    df_base = build_base_series_with_weather(df_raw, building, df_metadata, df_weather)

    n_total = len(df_base)
    train_end_idx = int(n_total * TRAIN_SPLIT)
    val_end_idx = int(n_total * (TRAIN_SPLIT + VAL_SPLIT))

    df_train_raw = df_base.iloc[:train_end_idx]
    df_val_raw = df_base.iloc[train_end_idx:val_end_idx]
    df_test_raw = df_base.iloc[val_end_idx:]

    mediana_cladire = df_train_raw['consumed_energy'].median()

    df_train = apply_split_cleaning_and_features(df_train_raw, mediana_cladire)
    df_val = apply_split_cleaning_and_features(df_val_raw, mediana_cladire)
    df_test = apply_split_cleaning_and_features(df_test_raw, mediana_cladire)

    if df_train.empty or df_val.empty or df_test.empty:
        return {
            'skip': True,
            'reason': f"Split gol dupa curatare (train={len(df_train)}, val={len(df_val)}, test={len(df_test)})"
        }

    min_required = LOOKBACK + FORECAST
    if len(df_train) < min_required or len(df_val) < min_required or len(df_test) < min_required:
        return {
            'skip': True,
            'reason': f"Split prea mic dupa curatare (train={len(df_train)}, val={len(df_val)}, test={len(df_test)})"
        }

    if df_train[features_to_scale].shape[0] == 0:
        return {
            'skip': True,
            'reason': "Split train fara randuri pentru scalare"
        }

    energy_scaler = MinMaxScaler()
    energy_scaler.fit(df_train[['consumed_energy']])

    scaler = MinMaxScaler()
    scaler.fit(df_train[features_to_scale])
    df_train.loc[:, features_to_scale] = scaler.transform(df_train[features_to_scale])
    df_val.loc[:, features_to_scale] = scaler.transform(df_val[features_to_scale])
    df_test.loc[:, features_to_scale] = scaler.transform(df_test[features_to_scale])

    X_train, y_train, ts_train = create_windows(df_train)
    X_val, y_val, ts_val = create_windows(df_val)
    X_test, y_test, ts_test = create_windows(df_test)

    if X_train.shape[0] == 0 or X_val.shape[0] == 0 or X_test.shape[0] == 0:
        return {
            'skip': True,
            'reason': f"Ferestre insuficiente (train={X_train.shape[0]}, val={X_val.shape[0]}, test={X_test.shape[0]})"
        }

    return {
        'scaler': scaler,
        'energy_scaler': energy_scaler,
        'splits': {
            'train': {'df': df_train, 'X_math': X_train, 'y': y_train, 'ts': ts_train},
            'val': {'df': df_val, 'X_math': X_val, 'y': y_val, 'ts': ts_val},
            'test': {'df': df_test, 'X_math': X_test, 'y': y_test, 'ts': ts_test}
        }
    }


In [ ]:
class GlobalTextConditionedModel(nn.Module):
    def __init__(self, num_math_features, text_emb_dim=768, hidden_dim=64, num_building_types=16):
        super().__init__()
        math_out_dim = hidden_dim * 2

        self.lstm = nn.LSTM(input_size=num_math_features, hidden_size=hidden_dim,
                            num_layers=1, batch_first=True, bidirectional=True)

        self.static_processor = nn.Sequential(
            nn.Linear(text_emb_dim, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, math_out_dim), nn.LayerNorm(math_out_dim)
        )
        self.dynamic_processor = nn.Sequential(
            nn.Linear(text_emb_dim, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, math_out_dim), nn.LayerNorm(math_out_dim)
        )

        self.film = nn.Linear(math_out_dim, math_out_dim * 2)
        nn.init.zeros_(self.film.weight)
        nn.init.zeros_(self.film.bias)
        with torch.no_grad():
            self.film.bias[:math_out_dim] = 1.0

        self.math_head = nn.Sequential(
            nn.Linear(math_out_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, FORECAST)
        )

        self.offset_head = nn.Sequential(
            nn.Linear(math_out_dim, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, FORECAST)
        )
        nn.init.zeros_(self.offset_head[-1].weight)
        nn.init.zeros_(self.offset_head[-1].bias)

        self.building_classifier = nn.Linear(math_out_dim, num_building_types)
        self._attn_scale = math_out_dim ** 0.5

    def _attention(self, query, lstm_seq):
        scores = torch.bmm(lstm_seq, query.unsqueeze(-1)).squeeze(-1)
        weights = torch.softmax(scores / self._attn_scale, dim=-1)
        context = torch.bmm(weights.unsqueeze(1), lstm_seq).squeeze(1)
        return context, weights

    def forward(self, math_input, static_emb, dynamic_emb):
        lstm_seq, (h_n, _) = self.lstm(math_input)
        math_vector = torch.cat((h_n[-2], h_n[-1]), dim=1)

        static_vec = self.static_processor(static_emb)
        dynamic_vec = self.dynamic_processor(dynamic_emb)

        gamma, beta = self.film(static_vec).chunk(2, dim=-1)
        math_cond = gamma * math_vector + beta
        math_pred = self.math_head(math_cond)

        context, _ = self._attention(dynamic_vec, lstm_seq)
        offset = self.offset_head(context)

        pred = math_pred + offset
        building_logits = self.building_classifier(static_vec)
        return pred, building_logits, math_pred, offset


In [ ]:
from transformers import AutoTokenizer, AutoModel
import os
import hashlib

model_name = "sentence-transformers/all-mpnet-base-v2"
print(f"Se incarca modelul: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
text_model = AutoModel.from_pretrained(model_name).to(device)
text_model.eval()

SITE_CLIMATE_MAP = {
    "Panther":  "humid subtropical (2A)",
    "Robin":    "mixed humid (4A)",
    "Fox":      "hot arid desert (2B)",
    "Rat":      "mixed humid (4A)",
    "Bear":     "dry temperate (3C)",
    "Lamb":     "mixed humid (4A)",
    "Eagle":    "mixed humid (4A)",
    "Moose":    "cold continental (6A)",
    "Gator":    "humid subtropical (2A)",
    "Bull":     "humid subtropical (2A)",
    "Bobcat":   "semi-arid (5B)",
    "Crow":     "cold continental (6A)",
    "Wolf":     "cool oceanic (5A)",
    "Hog":      "cold continental (6A)",
    "Peacock":  "humid continental (5A)",
    "Cockatoo": "cold continental (6A)",
    "Shrew":    "mixed humid (4A)",
    "Swan":     "dry temperate (3C)",
    "Mouse":    "mixed humid (4A)",
}

_HOLIDAY_CACHE = {}

def _country_from_tz(tz):
    if not tz:
        return None
    if tz.startswith("Europe/London"):
        return "GB"
    if tz.startswith("Europe/Dublin"):
        return "IE"
    if tz.startswith("US/"):
        return "US"
    if tz in ("America/Toronto", "America/Montreal", "America/Edmonton", "Canada/Eastern"):
        return "CA"
    if tz.startswith("America/"):
        return "US"
    return None

def _get_holiday_cal(country, year):
    key = (country, year)
    if key not in _HOLIDAY_CACHE:
        try:
            import holidays as _hol
            _HOLIDAY_CACHE[key] = _hol.country_holidays(country, years=[year - 1, year, year + 1])
        except Exception:
            _HOLIDAY_CACHE[key] = {}
    return _HOLIDAY_CACHE[key]

def _static_parts(meta):
    usage = meta.get("primaryspaceusage", "Unknown")
    year  = meta.get("yearbuilt", np.nan)
    floor_count = meta.get("numberoffloors", np.nan)

    area_m2 = meta.get("sqm", np.nan)
    if pd.isna(area_m2):
        area_ft2 = meta.get("sqft", np.nan)
        if pd.notna(area_ft2):
            area_m2 = float(area_ft2) * 0.092903

    def _size_class(a):
        if pd.isna(a): return "unknown size"
        return "small" if a < 5000 else ("medium" if a < 20000 else "large")

    site    = meta.get("site_id", "")
    climate = SITE_CLIMATE_MAP.get(str(site), "temperate")

    if pd.notna(year):
        y = int(year)
        if y < 1980:
            age_desc = "built before 1980 with poor insulation"
        elif y <= 2000:
            age_desc = "built between 1980 and 2000 with standard insulation"
        else:
            age_desc = "built after 2000 with efficient insulation"
    else:
        age_desc = "of unknown construction age"

    OCCUPANCY_DESC = {
        "Education":                    "university or school campus with weekday-heavy occupancy",
        "Office":                       "office building with typical 9-to-5 business hours",
        "Entertainment/publicassembly": "public assembly venue with variable occupancy",
        "Lodging/residential":          "residential or lodging building with continuous occupancy",
        "Public services":              "public services building with regular business hours",
        "Healthcare":                   "healthcare facility with 24-hour continuous operation",
        "Warehouse/storage":            "warehouse or storage facility with irregular occupancy",
        "Other":                        "mixed-use building",
    }
    occupancy_desc = OCCUPANCY_DESC.get(usage, f"{usage} building")
    size_desc = _size_class(area_m2)

    parts = [
        f"This is a {size_desc} {occupancy_desc},",
        f"{age_desc},",
        f"located in a {climate} climate.",
    ]
    if pd.notna(floor_count):
        floors = int(floor_count)
        parts.append(f"The building has {floors} floor{'s' if floors != 1 else ''}.")

    other_meters = [label for col, label in [
        ("chilledwater", "district chilled water"),
        ("steam", "district steam"),
        ("hotwater", "hot water"),
        ("gas", "gas"),
    ] if str(meta.get(col, "")).strip().lower() == "yes"]
    if other_meters:
        parts.append(
            "Beyond electricity, the building is also served by "
            + ", ".join(other_meters)
            + "; primary heating or cooling is partly off the electric meter, moderating its weather sensitivity."
        )
    else:
        parts.append("Electricity is the building's primary energy meter, capturing most heating and cooling load.")

    heating = meta.get("heatingtype", np.nan)
    if pd.notna(heating) and str(heating).strip():
        parts.append(f"Heating system: {str(heating).strip()}.")

    occ = meta.get("occupants", np.nan)
    if pd.notna(occ):
        try:
            parts.append(f"Design occupancy is about {int(float(occ))} people.")
        except Exception:
            pass

    return parts


def _dynamic_parts(meta, ts_pd, weather_df_site=None, lookback_energy=None):
    usage = meta.get("primaryspaceusage", "Unknown")
    month = ts_pd.month
    dow   = ts_pd.dayofweek

    if month in [12, 1, 2]:
        season_desc = "It is winter: heating demand is expected to be high."
    elif month in [3, 4, 5]:
        season_desc = "It is spring: energy demand is transitioning with moderate heating or cooling."
    elif month in [6, 7, 8]:
        season_desc = "It is summer: cooling demand is expected to dominate consumption."
    else:
        season_desc = "It is fall: energy demand is transitioning with variable heating and cooling."

    if dow >= 5:
        day_desc = "This is a weekend: occupancy is typically reduced or absent."
    elif dow == 0:
        day_desc = "This is Monday, start of the work week: occupancy ramps up from weekend lows."
    elif dow == 4:
        day_desc = "This is Friday, end of the work week: occupancy may decline toward the weekend."
    else:
        day_desc = "This is a mid-week workday: full occupancy and normal operational loads expected."

    parts = [season_desc, day_desc]

    tz = str(meta.get("timezone", ""))
    country = _country_from_tz(tz)
    if country is not None:
        try:
            hol = _get_holiday_cal(country, ts_pd.year)
            d0 = ts_pd.date()
            d1 = (ts_pd + pd.Timedelta(hours=FORECAST - 1)).date()
            hol_name = None
            for d in (d0, d1):
                if d in hol:
                    hol_name = hol.get(d)
                    break
            if hol_name:
                parts.append(f"The forecast period falls on a public holiday ({hol_name}): occupancy will be sharply reduced.")
        except Exception:
            pass
    if usage == "Education":
        if month in (6, 7) or (month == 8 and ts_pd.day <= 20):
            parts.append("The university is likely on summer break: campus occupancy is minimal.")
        elif (month == 12 and ts_pd.day >= 18) or (month == 1 and ts_pd.day <= 6):
            parts.append("The university is in winter recess: campus activity is reduced.")

    if weather_df_site is not None and not weather_df_site.empty and 'airTemperature' in weather_df_site.columns:
        try:
            ts_lookback_end   = ts_pd - pd.Timedelta(hours=1)
            ts_lookback_start = ts_pd - pd.Timedelta(hours=24)
            temp_series = weather_df_site.loc[ts_lookback_start:ts_lookback_end, 'airTemperature'].dropna()
            if len(temp_series) > 0:
                avg_temp = temp_series.mean()
                if avg_temp < 0:
                    temp_desc = f"very cold (avg {avg_temp:.0f}C)"
                elif avg_temp < 10:
                    temp_desc = f"cold (avg {avg_temp:.0f}C)"
                elif avg_temp < 18:
                    temp_desc = f"cool (avg {avg_temp:.0f}C)"
                elif avg_temp < 25:
                    temp_desc = f"mild (avg {avg_temp:.0f}C)"
                elif avg_temp < 32:
                    temp_desc = f"warm (avg {avg_temp:.0f}C)"
                else:
                    temp_desc = f"hot (avg {avg_temp:.0f}C)"
                parts.append(f"Recent outdoor temperature (last 24 hours): {temp_desc}.")
        except Exception:
            pass

    if weather_df_site is not None and not weather_df_site.empty and 'airTemperature' in weather_df_site.columns:
        try:
            fc = weather_df_site.loc[ts_pd:ts_pd + pd.Timedelta(hours=FORECAST - 1)]
            ftemp = fc['airTemperature'].dropna()
            if len(ftemp) > 0:
                favg = float(ftemp.mean())
                if favg < 0:    fdesc = f"very cold (avg {favg:.0f}C)"
                elif favg < 10: fdesc = f"cold (avg {favg:.0f}C)"
                elif favg < 18: fdesc = f"cool (avg {favg:.0f}C)"
                elif favg < 25: fdesc = f"mild (avg {favg:.0f}C)"
                elif favg < 32: fdesc = f"warm (avg {favg:.0f}C)"
                else:           fdesc = f"hot (avg {favg:.0f}C)"
                extras = []
                rec = weather_df_site.loc[ts_pd - pd.Timedelta(hours=24):ts_pd - pd.Timedelta(hours=1), 'airTemperature'].dropna()
                if len(rec) > 0:
                    delta = favg - float(rec.mean())
                    if delta >= 3:    extras.append("warming sharply")
                    elif delta <= -3: extras.append("cooling sharply")
                if 'windSpeed' in fc.columns:
                    w = fc['windSpeed'].dropna()
                    if len(w) > 0 and float(w.mean()) > 8:
                        extras.append("strong winds")
                if 'precipDepth1HR' in fc.columns:
                    p = fc['precipDepth1HR'].dropna()
                    p = p[p >= 0]
                    if len(p) > 0 and float(p.sum()) > 2:
                        extras.append("precipitation expected")
                tail = (", " + ", ".join(extras)) if extras else ""
                parts.append(f"Next 24h weather forecast: {fdesc}{tail}.")
        except Exception:
            pass

    if lookback_energy is not None and len(lookback_energy) >= 48:
        e = np.asarray(lookback_energy, dtype=np.float32)

        first24 = e[:24].mean()
        last24  = e[-24:].mean()
        wmean   = e.mean()

        if last24 > first24 * 1.15:
            trend = "rising trend over the past week"
        elif last24 < first24 * 0.85:
            trend = "declining trend over the past week"
        else:
            trend = "stable consumption over the past week"

        ratio = last24 / (wmean + 1e-9)
        if ratio > 1.2:
            recent = "notably above-average consumption in the last 24 hours"
        elif ratio < 0.8:
            recent = "notably below-average consumption in the last 24 hours"
        else:
            recent = "near-average consumption in the last 24 hours"

        cv = e.std() / (wmean + 1e-9)
        if cv > 0.5:
            variability = "highly variable load profile"
        elif cv > 0.25:
            variability = "moderately variable load profile"
        else:
            variability = "very consistent load profile"

        hourly_avg = np.array([e[h::24].mean() for h in range(24)])
        peak_h = int(np.argmax(hourly_avg))
        if 5 <= peak_h <= 9:
            peak = f"morning peak around {peak_h}:00"
        elif 10 <= peak_h <= 14:
            peak = f"midday peak around {peak_h}:00"
        elif 15 <= peak_h <= 20:
            peak = f"afternoon-evening peak around {peak_h}:00"
        else:
            peak = f"off-hours peak around {peak_h}:00"

        parts.append(
            f"Recent load profile: {trend}, {recent}, {variability}, with {peak}."
        )

        if len(e) >= 168:
            hour_index = pd.date_range(
                end=ts_pd - pd.Timedelta(hours=1), periods=len(e), freq='h'
            )
            we_mask = hour_index.dayofweek >= 5
            if we_mask.sum() > 0 and (~we_mask).sum() > 0:
                we_mean = float(e[we_mask].mean())
                wd_mean = float(e[~we_mask].mean())
                r = we_mean / (wd_mean + 1e-9)
                if r < 0.7:
                    occ = "significantly lower weekend consumption — occupancy-driven load"
                elif r > 1.3:
                    occ = "higher weekend consumption — non-office usage pattern"
                else:
                    occ = "similar weekday and weekend load — continuous operation"
                parts.append(f"Occupancy pattern: {occ}.")

    return parts


def generate_static_text(building_name, building_metadata_row):
    meta = building_metadata_row.iloc[0] if building_metadata_row is not None and not building_metadata_row.empty else {}
    return " ".join(_static_parts(meta))


def generate_dynamic_text(building_name, forecast_start_timestamp,
                          building_metadata_row, weather_df_site=None,
                          lookback_energy=None):
    meta = building_metadata_row.iloc[0] if building_metadata_row is not None and not building_metadata_row.empty else {}
    ts_pd = pd.Timestamp(forecast_start_timestamp)
    return " ".join(_dynamic_parts(meta, ts_pd, weather_df_site, lookback_energy))


def generate_expert_text(building_name, forecast_start_timestamp,
                          building_metadata_row, weather_df_site=None,
                          lookback_energy=None):
    s = generate_static_text(building_name, building_metadata_row)
    d = generate_dynamic_text(building_name, forecast_start_timestamp,
                              building_metadata_row, weather_df_site, lookback_energy)
    return (s + " " + d) if d else s


embedding_cache = {}
text_emb_dim = getattr(text_model.config, "hidden_size", 768)

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

MAX_TOKENS = 384

def get_embeddings_batch(texts, batch_size=64):
    if len(texts) == 0:
        return np.empty((0, text_emb_dim), dtype=np.float32)
    all_embs = []
    n_truncat = 0
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        lens = [len(ids) for ids in tokenizer(batch_texts, truncation=False)["input_ids"]]
        n_truncat += sum(1 for L in lens if L > MAX_TOKENS)
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True,
                           truncation=True, max_length=MAX_TOKENS).to(device)
        with torch.no_grad():
            outputs = text_model(**inputs)
        emb = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])
        all_embs.append(emb.cpu().numpy())
    if n_truncat:
        print(f"  [ATENTIE] {n_truncat} texte au depasit {MAX_TOKENS} tokeni si au fost trunchiate.")
    return np.vstack(all_embs)

EMBED_CACHE_DIR = "embeddings_cache"
TEXT_VERSION = "v10"
os.makedirs(EMBED_CACHE_DIR, exist_ok=True)

def get_or_create_embeddings(building, split_name, timestamps, X_math_windows,
                              building_metadata_row, df_weather_site,
                              scaler=None, batch_size=64):

    energy_scaled = X_math_windows[:, :, 0]
    if scaler is not None:
        dummy = np.zeros((energy_scaled.size, len(features_to_scale)))
        dummy[:, 0] = energy_scaled.flatten()
        energy_unscaled = scaler.inverse_transform(dummy)[:, 0].reshape(energy_scaled.shape)
    else:
        energy_unscaled = energy_scaled

    texts = [
        generate_expert_text(
            building, ts, building_metadata_row, df_weather_site,
            lookback_energy=energy_unscaled[i]
        )
        for i, ts in enumerate(timestamps)
    ]

    text_hash = hashlib.md5("\x1f".join(texts).encode("utf-8")).hexdigest()[:10]
    cache_path = os.path.join(
        EMBED_CACHE_DIR,
        f"{building}_{split_name}_mpnet_{TEXT_VERSION}_lb{LOOKBACK}_{text_hash}.npy"
    )
    if os.path.exists(cache_path):
        cached = np.load(cache_path)
        if len(cached) == len(timestamps):
            return cached
        print(f"  Cache invalid pentru {building}/{split_name} "
              f"(cached={len(cached)}, asteptat={len(timestamps)}). Regeneram...")

    embs = get_embeddings_batch(texts, batch_size=batch_size)
    np.save(cache_path, embs)
    return embs

print(f"Text version {TEXT_VERSION} activ — temperatura din ultimele 24h lookback.")


In [ ]:
import copy
import demo.notebook_export as _nx

results_dict = {}
baseline_preds_dict = {}
test_timestamps_dict = {}
baseline_lstm_weights = {}
baseline_math_weights = {}

import time
_t_ph_notext = time.perf_counter()
for building in TARGET_BUILDINGS:
    print(f"Procesare & Antrenament Baseline: {building}")

    if building not in precomputed_math:
        precomputed_math[building] = prepare_building_math_data(building)

    data_pack = precomputed_math[building]
    if data_pack.get('skip'):
        print(f"Skip {building}: {data_pack.get('reason', 'date insuficiente dupa curatare')}")
        continue

    scaler = data_pack['scaler']
    split_train = data_pack['splits']['train']
    split_val = data_pack['splits']['val']
    split_test = data_pack['splits']['test']

    X_train_raw, y_train_raw = split_train['X_math'], split_train['y']
    X_val_raw, y_val_raw = split_val['X_math'], split_val['y']
    X_test_raw, y_test_raw = split_test['X_math'], split_test['y']

    X_train = torch.tensor(X_train_raw, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train_raw, dtype=torch.float32).to(device)
    X_val = torch.tensor(X_val_raw, dtype=torch.float32).to(device)
    y_val = torch.tensor(y_val_raw, dtype=torch.float32).to(device)
    X_test = torch.tensor(X_test_raw, dtype=torch.float32).to(device)
    y_test = torch.tensor(y_test_raw, dtype=torch.float32).to(device)

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)

    model = BaselineLSTM(num_math_features=len(all_features)).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    print("Antrenament pornit cu Early Stopping...")
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_weights = None

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                preds = model(batch_X)
                val_loss += criterion(preds, batch_y).item()

        val_loss /= len(val_loader)
        train_loss /= len(train_loader)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoca [{epoch+1}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_weights = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print(f"Early Stopping la epoca {epoch+1}. Cel mai bun Val Loss: {best_val_loss:.4f}")
            break

    model.load_state_dict(best_model_weights)
    baseline_lstm_weights[building] = copy.deepcopy(model.lstm.state_dict())
    baseline_math_weights[building] = copy.deepcopy(model.fc.state_dict())

    print("Evaluare pe Test Set...")
    model.eval()
    with torch.no_grad():
        test_preds_scaled = model(X_test).cpu().numpy()
        y_test_scaled = y_test.cpu().numpy()

    energy_scaler = data_pack['energy_scaler']
    test_preds_actual = energy_scaler.inverse_transform(
        test_preds_scaled.reshape(-1, 1)
    ).reshape(test_preds_scaled.shape)
    y_test_actual = energy_scaler.inverse_transform(
        y_test_scaled.reshape(-1, 1)
    ).reshape(y_test_scaled.shape)

    mae, rmse, wape, r2, max_err = calculate_metrics(y_test_actual.flatten(), test_preds_actual.flatten())

    print(f"R2: {r2:.4f} | MAE: {mae:.2f} | WAPE: {wape:.2%}")
    results_dict[building] = {'MAE': mae, 'RMSE': rmse, 'WAPE': wape, 'R2': r2, 'Max_Err': max_err}

    test_timestamps_dict[building] = split_test['ts']
    baseline_preds_dict[building] = test_preds_actual



print(f"[timp] Antrenare BiLSTM per cladire (total {len(results_dict)} modele): {(time.perf_counter()-_t_ph_notext)/60:.1f} min")
_nx.save_perhouse_dicts(results_dict=results_dict)


In [ ]:
import copy
import os
import numpy as np
from torch.utils.data import Dataset, DataLoader

GLOBAL_TRAIN_STRIDE = 1
EMB_DTYPE = np.float16

def _gemb_path(building, tag, h):
    return os.path.join(EMBED_CACHE_DIR, f"{building}_{tag}_mpnet_{TEXT_VERSION}_lb{LOOKBACK}_{h}.npy")

def build_dynamic_emb(building, split_name, timestamps, X_math_windows, meta_row, dws, scaler):
    energy_scaled = X_math_windows[:, :, 0]
    dummy = np.zeros((energy_scaled.size, len(features_to_scale)))
    dummy[:, 0] = energy_scaled.flatten()
    energy_unscaled = scaler.inverse_transform(dummy)[:, 0].reshape(energy_scaled.shape)
    texts = [generate_dynamic_text(building, ts, meta_row, dws, lookback_energy=energy_unscaled[i])
             for i, ts in enumerate(timestamps)]
    h = hashlib.md5("\x1f".join(texts).encode("utf-8")).hexdigest()[:10]
    cp = _gemb_path(building, f"dyn_{split_name}", h)
    if os.path.exists(cp):
        e = np.load(cp)
        if len(e) == len(texts):
            return e.astype(EMB_DTYPE)
    e = get_embeddings_batch(texts, batch_size=64).astype(EMB_DTYPE)
    np.save(cp, e)
    return e

def build_static_emb(building, meta_row):
    txt = generate_static_text(building, meta_row)
    h = hashlib.md5(txt.encode("utf-8")).hexdigest()[:10]
    cp = _gemb_path(building, "static", h)
    if os.path.exists(cp):
        return np.load(cp).astype(EMB_DTYPE)
    e = get_embeddings_batch([txt], batch_size=1)[0].astype(EMB_DTYPE)
    np.save(cp, e)
    return e

unique_types = sorted(df_metadata['primaryspaceusage'].dropna().unique().tolist())
type_to_idx = {t: i for i, t in enumerate(unique_types)}
num_building_types = len(unique_types)

global_store = {}
valid_buildings = []
import time
_t_emb = time.perf_counter()
for b in TARGET_BUILDINGS:
    if b not in precomputed_math:
        precomputed_math[b] = prepare_building_math_data(b)
    dp = precomputed_math[b]
    if dp.get('skip'):
        print(f"  Skip {b}: {dp.get('reason', '')}")
        continue
    scaler = dp['scaler']
    meta_row = df_metadata[df_metadata['building_id'] == b]
    dws = None
    if not meta_row.empty:
        sid = meta_row.iloc[0].get('site_id', None)
        if sid is not None:
            dws = df_weather[df_weather['site_id'] == sid].copy()
            if not dws.empty and 'timestamp' in dws.columns:
                dws.set_index('timestamp', inplace=True)
    btype_label = str(meta_row.iloc[0].get('primaryspaceusage', 'Unknown')) if not meta_row.empty else 'Unknown'
    static_e = build_static_emb(b, meta_row)
    splits = {}
    for sp in ['train', 'val', 'test']:
        s = dp['splits'][sp]
        dyn = build_dynamic_emb(b, sp, s['ts'], s['X_math'], meta_row, dws, scaler)
        splits[sp] = {'X': s['X_math'], 'y': s['y'], 'dyn': dyn, 'ts': s['ts']}
    global_store[b] = {'static': static_e, 'type': type_to_idx.get(btype_label, 0),
                       'splits': splits, 'energy_scaler': dp['energy_scaler']}
    valid_buildings.append(b)

print(f"[timp] Generare embeddings + global_store ({len(valid_buildings)} cladiri): {(time.perf_counter()-_t_emb)/60:.1f} min")
class GlobalWindowDataset(Dataset):
    def __init__(self, buildings, split, stride=1):
        self.split = split
        self.index = []
        for b in buildings:
            n = len(global_store[b]['splits'][split]['y'])
            self.index.extend((b, i) for i in range(0, n, stride))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, k):
        b, i = self.index[k]
        s = global_store[b]['splits'][self.split]
        math = torch.tensor(s['X'][i], dtype=torch.float32)
        dyn = torch.tensor(np.asarray(s['dyn'][i], dtype=np.float32))
        stat = torch.tensor(np.asarray(global_store[b]['static'], dtype=np.float32))
        y = torch.tensor(s['y'][i], dtype=torch.float32)
        return math, stat, dyn, y, global_store[b]['type']


train_ds = GlobalWindowDataset(valid_buildings, 'train', stride=GLOBAL_TRAIN_STRIDE)
val_ds = GlobalWindowDataset(valid_buildings, 'val', stride=1)
print(f"Ferestre train: {len(train_ds):,} | val: {len(val_ds):,}")

pin = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0, pin_memory=pin)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=0, pin_memory=pin)

global_model = GlobalTextConditionedModel(num_math_features=len(all_features),
                                          num_building_types=num_building_types).to(device)
criterion_mse = nn.MSELoss()
criterion_cls = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(global_model.parameters(), lr=LR)
AUX_LAMBDA = 0.1

GLOBAL_CKPT = f"global_model_{TEXT_VERSION}.pt"
_need_train = not os.path.exists(GLOBAL_CKPT)

if not _need_train:
    global_model.load_state_dict(torch.load(GLOBAL_CKPT, map_location=device))

best_val = float('inf'); patience_counter = 0; best_w = None
_t_train = time.perf_counter()
for epoch in range(EPOCHS if _need_train else 0):
    global_model.train(); tr = 0.0
    for math_b, stat_b, dyn_b, y_b, lbl_b in train_loader:
        math_b = math_b.to(device, non_blocking=True); stat_b = stat_b.to(device, non_blocking=True)
        dyn_b = dyn_b.to(device, non_blocking=True); y_b = y_b.to(device, non_blocking=True)
        lbl_b = lbl_b.to(device, non_blocking=True)
        optimizer.zero_grad()
        pred, logits, _, off = global_model(math_b, stat_b, dyn_b)
        loss = criterion_mse(pred, y_b) + AUX_LAMBDA * criterion_cls(logits, lbl_b) + 0.01 * off.abs().mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(global_model.parameters(), max_norm=1.0)
        optimizer.step()
        tr += criterion_mse(pred, y_b).item()
    global_model.eval(); vl = 0.0
    with torch.no_grad():
        for math_b, stat_b, dyn_b, y_b, lbl_b in val_loader:
            math_b = math_b.to(device, non_blocking=True); stat_b = stat_b.to(device, non_blocking=True)
            dyn_b = dyn_b.to(device, non_blocking=True); y_b = y_b.to(device, non_blocking=True)
            pred, _, _, _ = global_model(math_b, stat_b, dyn_b)
            vl += criterion_mse(pred, y_b).item()
    vl /= max(len(val_loader), 1); tr /= max(len(train_loader), 1)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoca {epoch+1}/{EPOCHS}: train={tr:.4f} val={vl:.4f}")
    if vl < best_val:
        best_val = vl; patience_counter = 0; best_w = copy.deepcopy(global_model.state_dict())
    else:
        patience_counter += 1
    if patience_counter >= PATIENCE:
        print(f"  Early stopping la epoca {epoch+1}")
        break
if _need_train:
    if best_w is not None:
        global_model.load_state_dict(best_w)
    torch.save(global_model.state_dict(), GLOBAL_CKPT)
    print(f"[timp] Antrenare Hibrid global: {(time.perf_counter()-_t_train)/60:.1f} min")


In [ ]:
results_hybrid_dict = {}
all_test_embeddings, all_test_timestamps, all_test_building_labels = [], [], []
idx_to_type = {i: t for t, i in type_to_idx.items()}

def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)
pin_mem = torch.cuda.is_available()

def _ph_loader(sp, static_vec, shuffle):
    n = len(sp['y'])
    Xm  = to_tensor(np.asarray(sp['X'],   dtype=np.float32))
    Xdy = to_tensor(np.asarray(sp['dyn'], dtype=np.float32))
    Xst = to_tensor(np.repeat(static_vec[None, :], n, axis=0))
    yy  = to_tensor(np.asarray(sp['y'],   dtype=np.float32))
    return DataLoader(TensorDataset(Xm, Xst, Xdy, yy), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=0, pin_memory=pin_mem)

import time
_t_ph_text = time.perf_counter()
for building in valid_buildings:
    gs = global_store[building]
    sp_tr, sp_va, sp_te = gs['splits']['train'], gs['splits']['val'], gs['splits']['test']
    if len(sp_tr['y']) == 0 or len(sp_va['y']) == 0 or len(sp_te['y']) == 0:
        print(f"Skip {building}: split gol"); continue

    static_vec = np.asarray(gs['static'], dtype=np.float32).ravel()
    train_loader = _ph_loader(sp_tr, static_vec, True)
    val_loader   = _ph_loader(sp_va, static_vec, False)

    model_h = GlobalTextConditionedModel(num_math_features=len(all_features),
                                         num_building_types=num_building_types).to(device)
    criterion_mse = nn.MSELoss()
    optimizer = torch.optim.Adam(model_h.parameters(), lr=LR)

    best_val, patience, best_w = float('inf'), 0, None
    for epoch in range(EPOCHS):
        model_h.train()
        for bm, bs, bd, by in train_loader:
            bm, bs, bd, by = bm.to(device), bs.to(device), bd.to(device), by.to(device)
            optimizer.zero_grad()
            pred, _logits, _base, offset = model_h(bm, bs, bd)
            loss = criterion_mse(pred, by) + 0.01 * offset.abs().mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_h.parameters(), max_norm=1.0)
            optimizer.step()
        model_h.eval(); vloss = 0.0
        with torch.no_grad():
            for bm, bs, bd, by in val_loader:
                bm, bs, bd, by = bm.to(device), bs.to(device), bd.to(device), by.to(device)
                pred, _, _, _ = model_h(bm, bs, bd)
                vloss += criterion_mse(pred, by).item()
        vloss /= max(1, len(val_loader))
        if vloss < best_val:
            best_val, patience, best_w = vloss, 0, copy.deepcopy(model_h.state_dict())
        else:
            patience += 1
            if patience >= PATIENCE:
                break
    if best_w is not None:
        model_h.load_state_dict(best_w)

    model_h.eval()
    with torch.no_grad():
        n_te = len(sp_te['y'])
        Xm  = to_tensor(np.asarray(sp_te['X'],   dtype=np.float32)).to(device)
        Xdy = to_tensor(np.asarray(sp_te['dyn'], dtype=np.float32)).to(device)
        Xst = to_tensor(np.repeat(static_vec[None, :], n_te, axis=0)).to(device)
        pred_scaled, _, _, _ = model_h(Xm, Xst, Xdy)
        pred_scaled = pred_scaled.cpu().numpy()

    es = gs['energy_scaler']
    y_te = np.asarray(sp_te['y'], dtype=np.float32)
    pred_kwh = es.inverse_transform(pred_scaled.reshape(-1, 1)).reshape(pred_scaled.shape)
    y_kwh    = es.inverse_transform(y_te.reshape(-1, 1)).reshape(y_te.shape)
    mae, rmse, wape, r2, max_err = calculate_metrics(y_kwh.flatten(), pred_kwh.flatten())
    results_hybrid_dict[building] = {'MAE': mae, 'RMSE': rmse, 'WAPE': wape, 'R2': r2, 'Max_Err': max_err}

    all_test_embeddings.append(np.asarray(sp_te['dyn'], dtype=np.float32))
    all_test_timestamps.extend(list(sp_te['ts']))
    all_test_building_labels.extend([idx_to_type.get(gs['type'], 'Unknown')] * n_te)

    print(f"{building:<26} RMSE={rmse:.2f} | MAE={mae:.2f} | WAPE={wape:.2%} | R2={r2:.4f}")

print(f"[timp] Antrenare Hibrid per cladire (total {len(results_hybrid_dict)} modele): {(time.perf_counter()-_t_ph_text)/60:.1f} min")
_nx.save_perhouse_dicts(results_hybrid_dict=results_hybrid_dict)


In [ ]:
def _eval_global(model, b):
    s = global_store[b]['splits']['test']
    n = len(s['y'])
    model.eval()
    with torch.no_grad():
        math_b = torch.tensor(s['X'], dtype=torch.float32).to(device)
        dyn_b = torch.tensor(np.asarray(s['dyn'], dtype=np.float32)).to(device)
        stat_one = np.asarray(global_store[b]['static'], dtype=np.float32)
        stat_b = torch.tensor(np.broadcast_to(stat_one, (n, stat_one.shape[0])).copy()).to(device)
        pred, _, _, _ = model(math_b, stat_b, dyn_b)
        pred = pred.cpu().numpy()
    es = global_store[b]['energy_scaler']
    pred_a = es.inverse_transform(pred.reshape(-1, 1)).reshape(pred.shape)
    y_a = es.inverse_transform(s['y'].reshape(-1, 1)).reshape(s['y'].shape)
    return calculate_metrics(y_a.flatten(), pred_a.flatten())

results_global_dict = {}
for b in valid_buildings:
    mae, rmse, wape, r2, mx = _eval_global(global_model, b)
    results_global_dict[b] = {'MAE': mae, 'RMSE': rmse, 'WAPE': wape, 'R2': r2, 'Max_Err': mx}
print(f"Model global pur: RMSE mediu = {np.mean([v['RMSE'] for v in results_global_dict.values()]):.2f}")


In [ ]:
import copy

NOTEXT_CKPT = f"global_notext_{TEXT_VERSION}.pt"
notext_model = BaselineLSTM(num_math_features=len(all_features)).to(device)
crit_nt = nn.MSELoss()

_need_nt = not os.path.exists(NOTEXT_CKPT)
if not _need_nt:
    notext_model.load_state_dict(torch.load(NOTEXT_CKPT, map_location=device))
else:
    opt_nt = torch.optim.Adam(notext_model.parameters(), lr=LR)
    _pin_nt = torch.cuda.is_available()
    train_loader = DataLoader(GlobalWindowDataset(valid_buildings, 'train', stride=GLOBAL_TRAIN_STRIDE),
                              batch_size=256, shuffle=True, num_workers=0, pin_memory=_pin_nt)
    val_loader = DataLoader(GlobalWindowDataset(valid_buildings, 'val', stride=1),
                            batch_size=256, shuffle=False, num_workers=0, pin_memory=_pin_nt)

best_val_nt = float('inf'); pc_nt = 0; bw_nt = None
import time
_t_nt_train = time.perf_counter()
for epoch in range(EPOCHS if _need_nt else 0):
    notext_model.train(); tr = 0.0
    for math_b, stat_b, dyn_b, y_b, lbl_b in train_loader:
        math_b = math_b.to(device, non_blocking=True); y_b = y_b.to(device, non_blocking=True)
        opt_nt.zero_grad()
        pred = notext_model(math_b)
        loss = crit_nt(pred, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(notext_model.parameters(), max_norm=1.0)
        opt_nt.step()
        tr += loss.item()
    notext_model.eval(); vl = 0.0
    with torch.no_grad():
        for math_b, stat_b, dyn_b, y_b, lbl_b in val_loader:
            math_b = math_b.to(device, non_blocking=True); y_b = y_b.to(device, non_blocking=True)
            vl += crit_nt(notext_model(math_b), y_b).item()
    vl /= max(len(val_loader), 1); tr /= max(len(train_loader), 1)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoca {epoch+1}/{EPOCHS}: train={tr:.4f} val={vl:.4f}")
    if vl < best_val_nt:
        best_val_nt = vl; pc_nt = 0; bw_nt = copy.deepcopy(notext_model.state_dict())
    else:
        pc_nt += 1
    if pc_nt >= PATIENCE:
        print(f"  Early stopping la epoca {epoch+1}")
        break
if _need_nt:
    if bw_nt is not None:
        notext_model.load_state_dict(bw_nt)
    torch.save(notext_model.state_dict(), NOTEXT_CKPT)
    print(f"[timp] Antrenare BiLSTM global (notext): {(time.perf_counter()-_t_nt_train)/60:.1f} min")

def _eval_notext(model, b):
    s = global_store[b]['splits']['test']
    model.eval()
    with torch.no_grad():
        mb = torch.tensor(s['X'], dtype=torch.float32).to(device)
        pred = model(mb).cpu().numpy()
    es = global_store[b]['energy_scaler']
    pa = es.inverse_transform(pred.reshape(-1, 1)).reshape(pred.shape)
    ya = es.inverse_transform(s['y'].reshape(-1, 1)).reshape(s['y'].shape)
    return calculate_metrics(ya.flatten(), pa.flatten())

results_notext_dict = {}
for b in valid_buildings:
    mae, rmse, wape, r2, mx = _eval_notext(notext_model, b)
    results_notext_dict[b] = {'MAE': mae, 'RMSE': rmse, 'WAPE': wape, 'R2': r2, 'Max_Err': mx}

nt = np.mean([results_notext_dict[b]['RMSE'] for b in valid_buildings])
gt = np.mean([results_global_dict[b]['RMSE'] for b in valid_buildings])


print(f"  Global FARA text  RMSE : {nt:.2f}")
print(f"  Global CU text                RMSE : {gt:.2f}")
print(f"  Contributia TEXTULUI               : {nt - gt:+.2f} kWh ({100*(nt-gt)/nt:+.1f}%)")
wins_text = sum(1 for b in valid_buildings if results_global_dict[b]['RMSE'] < results_notext_dict[b]['RMSE'])
print(f"  Textul ajuta la {wins_text}/{len(valid_buildings)} cladiri")
if 'results_dict' in globals() and results_dict:
    common = [b for b in valid_buildings if b in results_dict]
    if common:
        base = np.mean([results_dict[b]['RMSE'] for b in common])
        print(f"  (Referinta: baseline per-cladire = {base:.2f})")
print("=" * 64)


In [ ]:
import os
import copy
from sota_baselines import PatchTST, PatchTSTConfig

torch.manual_seed(SEED)
np.random.seed(SEED)
pin = torch.cuda.is_available()

pt_cfg = PatchTSTConfig(seq_len=LOOKBACK, pred_len=FORECAST, enc_in=len(all_features))
pt_model = PatchTST(pt_cfg).to(device)
pt_buildings = valid_buildings

pt_train_ds = GlobalWindowDataset(pt_buildings, 'train', stride=GLOBAL_TRAIN_STRIDE)
pt_val_ds = GlobalWindowDataset(pt_buildings, 'val', stride=1)
pt_train_loader = DataLoader(pt_train_ds, batch_size=256, shuffle=True, num_workers=0, pin_memory=pin)
pt_val_loader = DataLoader(pt_val_ds, batch_size=256, shuffle=False, num_workers=0, pin_memory=pin)


pt_criterion = nn.MSELoss()
pt_optimizer = torch.optim.Adam(pt_model.parameters(), lr=LR)

PATCHTST_CKPT = f"patchtst_{TEXT_VERSION}.pt"
_need_pt = not os.path.exists(PATCHTST_CKPT)
if not _need_pt:
    pt_model.load_state_dict(torch.load(PATCHTST_CKPT, map_location=device))

best_pt = float('inf'); pc_pt = 0; bw_pt = None
import time
_t_pt_train = time.perf_counter()
for epoch in range(EPOCHS if _need_pt else 0):
    pt_model.train(); tr = 0.0
    for math_b, stat_b, dyn_b, y_b, lbl_b in pt_train_loader:
        math_b = math_b.to(device, non_blocking=True); y_b = y_b.to(device, non_blocking=True)
        pt_optimizer.zero_grad()
        pred = pt_model(math_b)[..., 0]
        loss = pt_criterion(pred, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(pt_model.parameters(), max_norm=1.0)
        pt_optimizer.step()
        tr += loss.item()
    pt_model.eval(); vl = 0.0
    with torch.no_grad():
        for math_b, stat_b, dyn_b, y_b, lbl_b in pt_val_loader:
            math_b = math_b.to(device, non_blocking=True); y_b = y_b.to(device, non_blocking=True)
            vl += pt_criterion(pt_model(math_b)[..., 0], y_b).item()
    vl /= max(len(pt_val_loader), 1); tr /= max(len(pt_train_loader), 1)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoca {epoch+1}/{EPOCHS}: train={tr:.4f} val={vl:.4f}")
    if vl < best_pt:
        best_pt = vl; pc_pt = 0; bw_pt = copy.deepcopy(pt_model.state_dict())
    else:
        pc_pt += 1
    if pc_pt >= PATIENCE:
        print(f"  Early stopping la epoca {epoch+1}")
        break
if _need_pt:
    if bw_pt is not None:
        pt_model.load_state_dict(bw_pt)
    torch.save(pt_model.state_dict(), PATCHTST_CKPT)
    print(f"[timp] Antrenare PatchTST: {(time.perf_counter()-_t_pt_train)/60:.1f} min")


def _eval_patchtst(model, b):
    s = global_store[b]['splits']['test']
    model.eval()
    with torch.no_grad():
        mb = torch.tensor(s['X'], dtype=torch.float32).to(device)
        pred = model(mb)[..., 0].cpu().numpy()
    es = global_store[b]['energy_scaler']
    pa = es.inverse_transform(pred.reshape(-1, 1)).reshape(pred.shape)
    ya = es.inverse_transform(s['y'].reshape(-1, 1)).reshape(s['y'].shape)
    return calculate_metrics(ya.flatten(), pa.flatten())


In [ ]:
import umap
import plotly.express as px
import pandas as pd
import numpy as np
import hashlib

def vizualizeaza_embeddinguri_3d_multidim(X_text_array, labels_building_type, timestamps_list):
    def get_season(ts):
        m = pd.Timestamp(ts).month
        if m in [12, 1, 2]:  return "Iarna"
        elif m in [3, 4, 5]: return "Primavara"
        elif m in [6, 7, 8]: return "Vara"
        else:                 return "Toamna"

    def get_day_type(ts):
        dow = pd.Timestamp(ts).dayofweek
        if dow >= 5:   return "Weekend"
        elif dow == 0: return "Luni"
        elif dow == 4: return "Vineri"
        else:          return "Miez saptamana"

    n_total = len(X_text_array)
    print(f"Total embeddings colectate: {n_total:,}")

    seasons   = [get_season(ts)   for ts in timestamps_list]
    day_types = [get_day_type(ts) for ts in timestamps_list]

    print("Calculam hash-uri pentru deduplicare...")
    seen_hashes, unique_idx = set(), []
    for i in range(n_total):
        h = hashlib.md5(X_text_array[i].astype(np.float32).tobytes()).hexdigest()
        if h not in seen_hashes:
            seen_hashes.add(h)
            unique_idx.append(i)

    X_unique        = X_text_array[unique_idx]
    labels_unique   = [labels_building_type[i] for i in unique_idx]
    seasons_unique  = [seasons[i]   for i in unique_idx]
    daytypes_unique = [day_types[i] for i in unique_idx]
    ts_unique       = [timestamps_list[i] for i in unique_idx]

    print(f"Embeddings unice: {len(X_unique):,} (din {n_total:,} total)")
    print(f"UMAP 3D pe {len(X_unique)} puncte... (dureaza ~5-15 secunde)")

    reducer = umap.UMAP(
        n_components=3, n_neighbors=15, min_dist=0.15,
        random_state=42, metric='cosine'
    )
    emb3d = reducer.fit_transform(X_unique)
    print("UMAP gata. Generam graficele...\n")

    df = pd.DataFrame({
        'X': emb3d[:, 0], 'Y': emb3d[:, 1], 'Z': emb3d[:, 2],
        'Tip_Cladire': labels_unique,
        'Sezon':       seasons_unique,
        'Tip_Zi':      daytypes_unique,
        'Data':        [pd.Timestamp(ts).strftime('%Y-%m-%d') for ts in ts_unique],
    })
    df['Hover'] = (
        '<b>' + df['Tip_Cladire'] + '</b><br>' +
        'Sezon: ' + df['Sezon'] + '<br>' +
        'Zi: '   + df['Tip_Zi'] + '<br>' +
        df['Data']
    )

    common = dict(
        x='X', y='Y', z='Z', hover_name='Hover',
        template="plotly_dark", width=1050, height=720,
        labels={'X': 'UMAP-1', 'Y': 'UMAP-2', 'Z': 'UMAP-3'}
    )

    print(">>> Grafic 1: Tip cladire")
    fig1 = px.scatter_3d(
        df.sort_values('Tip_Cladire'), color='Tip_Cladire',
        title="Spatiul Semantic 3D al Textului (MPNet + UMAP) — Tip cladire",
        **common
    )
    fig1.update_traces(marker=dict(size=6, opacity=0.85, line=dict(width=0)))
    fig1.update_layout(legend=dict(title="Tip cladire:", font=dict(size=12)),
                       margin=dict(l=0, r=0, b=0, t=50))
    fig1.show()

    print(">>> Grafic 2: Sezon")
    season_colors = {
        "Iarna":     "#4FC3F7",
        "Primavara": "#81C784",
        "Vara":      "#FFB74D",
        "Toamna":    "#FF7043",
    }
    fig2 = px.scatter_3d(
        df, color='Sezon',
        color_discrete_map=season_colors,
        category_orders={'Sezon': ["Iarna", "Primavara", "Vara", "Toamna"]},
        title="Spatiul Semantic 3D al Textului (MPNet + UMAP) — Sezon",
        **common
    )
    fig2.update_traces(marker=dict(size=6, opacity=0.85, line=dict(width=0)))
    fig2.update_layout(legend=dict(title="Sezon:", font=dict(size=12)),
                       margin=dict(l=0, r=0, b=0, t=50))
    fig2.show()

    print(">>> Grafic 3: Tip zi")
    daytype_colors = {
        "Weekend":        "#E91E63",
        "Luni":           "#9C27B0",
        "Vineri":         "#FF9800",
        "Miez saptamana": "#4CAF50",
    }
    fig3 = px.scatter_3d(
        df, color='Tip_Zi',
        color_discrete_map=daytype_colors,
        category_orders={'Tip_Zi': ["Miez saptamana", "Luni", "Vineri", "Weekend"]},
        title="Spatiul Semantic 3D al Textului (MPNet + UMAP) — Tip zi",
        **common
    )
    fig3.update_traces(marker=dict(size=6, opacity=0.85, line=dict(width=0)))
    fig3.update_layout(legend=dict(title="Tip zi:", font=dict(size=12)),
                       margin=dict(l=0, r=0, b=0, t=50))
    fig3.show()

    print(f"\nTotal puncte afisate: {len(X_unique)}")
    return df


if 'all_test_embeddings' in globals() and len(all_test_embeddings) > 0:
    X_all = np.vstack(all_test_embeddings)
    df_umap = vizualizeaza_embeddinguri_3d_multidim(
        X_all, all_test_building_labels, all_test_timestamps
    )
else:
    print("Embeddings agregate indisponibile. Ruleaza mai intai pipeline-ul hibrid.")

In [ ]:
import importlib, demo.notebook_export as nx
importlib.reload(nx)
nx.save_global_predictions(global_store)

In [ ]:
import importlib, demo.notebook_export as nx
importlib.reload(nx)
nx.save_embeddings(valid_buildings, df_metadata, build_static_emb, generate_static_text)

In [ ]:
import importlib, demo.notebook_export as nx
importlib.reload(nx)
nx.save_infer_bundles(valid_buildings, global_store, generate_static_text, df_metadata)

In [ ]:
import importlib, demo.notebook_export as nx
importlib.reload(nx)
nx.save_patchtst(pt_model, pt_buildings, global_store, device)

In [ ]:
import importlib, demo.notebook_export as nx
importlib.reload(nx)
nx.save_platform_metrics(
    results_dict=globals().get('results_dict'),
    results_hybrid_dict=globals().get('results_hybrid_dict'),
    results_notext_dict=globals().get('results_notext_dict'),
    results_global_dict=globals().get('results_global_dict'),
    results_patchtst_dict=globals().get('results_patchtst_dict'))

In [ ]:
import importlib, demo.notebook_export as nx
importlib.reload(nx)
nx.save_metrics_24h(global_store,
    results_dict=globals().get('results_dict'),
    results_hybrid_dict=globals().get('results_hybrid_dict'))